# 1. Validation of the MACCS Keys–PLS model

Reproduces the validation section of the Supporting Information:

| Output of this notebook | Corresponding item in the SI |
|---|---|
| Effective dimensionality of the descriptor space | SI Section 4 (1) |
| `output/table_s5.csv` | Table S5 |
| `output/table_s6.csv` | Table S6 |
| `output/figure_s1.png` | Figure S1 (a)–(c) |

**Input:** `data/solvent.csv` with columns `name`, `smiles`, `exp_yield`.
Rows whose `exp_yield` is empty are candidate solvents and are ignored here;
only the ten solvents with measured yields are used.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem.MACCSkeys import GenMACCSKeys
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

# Resolve paths so that the notebook runs both from the repository root
# and from inside notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

N_COMPONENTS = 3      # number of PLS latent variables used throughout the paper
RANDOM_SEED = 1       # seed for the y-randomization test

def maccs_matrix(smiles_list):
    """Convert a list of SMILES into a (n_molecules, 167) MACCS Keys matrix."""
    mols = [Chem.MolFromSmiles(s) for s in smiles_list]
    if any(m is None for m in mols):
        bad = [s for s, m in zip(smiles_list, mols) if m is None]
        raise ValueError(f"RDKit could not parse: {bad}")
    return np.array([GenMACCSKeys(m) for m in mols], dtype=float)

print("root:", ROOT)

## Load the training data

Only the ten solvents with measured yields enter the validation. The MACCS Keys
fingerprint is computed from the SMILES string with RDKit, giving a 167-bit
binary vector per solvent (bit 0 is a padding bit that is always zero).

In [ ]:
solvent = pd.read_csv(DATA / "solvent.csv")
train = solvent.dropna(subset=["exp_yield"]).reset_index(drop=True)

X = maccs_matrix(train["smiles"])           # (10, 167) binary matrix
y = train[["exp_yield"]].to_numpy(float)    # (10, 1) observed yields

print(f"{len(train)} training solvents, descriptor matrix {X.shape}")
train

## 1. Effective dimensionality of the descriptor space

The nominal ratio of 167 descriptors to 10 observations looks hopeless, but most
of the bits carry no information for this particular set of solvents:

* a bit that takes the **same value in all ten solvents** has zero variance and is
  reduced to zero by standardization;
* among the remaining bits, many are **identical or exactly complementary** to one
  another within these ten molecules, so they act as a single column;
* the centred design matrix can have a rank of at most `n - 1 = 9`.

Two columns are treated as the same pattern if one is equal to the other or to its
complement; this is why each pattern is canonicalized with `min(c, 1 - c)` below.

In [ ]:
is_constant = X.min(axis=0) == X.max(axis=0)
variable_idx = np.where(~is_constant)[0]

patterns = set()
for j in variable_idx:
    col = tuple(X[:, j].astype(int))
    # a column and its complement carry the same information here
    patterns.add(min(col, tuple(1 - np.array(col))))

rank = np.linalg.matrix_rank(X - X.mean(axis=0))

print(f"constant bits (no information) : {is_constant.sum()}")
print(f"variable bits                  : {len(variable_idx)}")
print(f"distinct column patterns       : {len(patterns)}")
print(f"rank of the centred matrix     : {rank}  (maximum possible = {len(train) - 1})")

## 2. Leave-one-out cross-validation (Table S5, Table S6)

In each fold one solvent is withheld, **both scalers are refitted on the remaining
nine solvents**, and the PLS model is retrained. Refitting the scalers inside the
fold is essential: fitting them once on all ten solvents would leak information
about the withheld sample into its own prediction and give an optimistic error.

`Q2` is the cross-validated coefficient of determination, computed against the
total sum of squares of the observed yields, so `Q2 = 0` corresponds to a model
that always returns the mean observed yield.

In [ ]:
def loo_predict(X, y, n_components):
    """Leave-one-out cross-validated predictions, scalers refitted in each fold."""
    preds = np.empty(len(y))
    for i in range(len(y)):
        keep = np.arange(len(y)) != i
        x_scaler, y_scaler = StandardScaler(), StandardScaler()
        X_tr = x_scaler.fit_transform(X[keep])
        y_tr = y_scaler.fit_transform(y[keep])
        pls = PLSRegression(n_components).fit(X_tr, y_tr)
        pred_scaled = pls.predict(x_scaler.transform(X[i:i + 1]))
        preds[i] = y_scaler.inverse_transform(pred_scaled)[0, 0]
    return preds


y_obs = y.ravel()
tss = ((y_obs - y_obs.mean()) ** 2).sum()
null_rmse = np.sqrt(((y_obs - y_obs.mean()) ** 2).mean())

rows, loo_preds = [], {}
for n in range(1, 6):
    p = loo_predict(X, y, n)
    loo_preds[n] = p
    rows.append({
        "n_components": n,
        "RMSE_LOO (%)": np.sqrt(((p - y_obs) ** 2).mean()),
        "Q2": 1 - ((p - y_obs) ** 2).sum() / tss,
        "Spearman rho": spearmanr(y_obs, p).statistic,
    })
rows.append({"n_components": "mean-only model", "RMSE_LOO (%)": null_rmse, "Q2": 0.0,
             "Spearman rho": np.nan})

table_s5 = pd.DataFrame(rows).round({"RMSE_LOO (%)": 1, "Q2": 2, "Spearman rho": 2})
table_s5.to_csv(OUT / "table_s5.csv", index=False)
table_s5

In [ ]:
table_s6 = pd.DataFrame({
    "solvent": train["name"],
    "observed (%)": y_obs,
    "LOO prediction (%), 1 component": loo_preds[1].round(1),
    f"LOO prediction (%), {N_COMPONENTS} components": loo_preds[N_COMPONENTS].round(1),
})
table_s6.to_csv(OUT / "table_s6.csv", index=False)
table_s6

## 3. y-Randomization

The observed yields are randomly permuted among the ten solvents and the **entire**
modelling procedure — standardization, PLS fitting and leave-one-out
cross-validation — is repeated 500 times. If the unpermuted model were merely
fitting noise, its error would be indistinguishable from the permuted ones.

`p` is the fraction of permutations that reach an error at least as low as the
unpermuted model. Because the permutations are random, the mean and standard
deviation depend on `RANDOM_SEED`; the value used for the paper is set at the top
of this notebook.

In [ ]:
def loo_rmse(X, y, n_components):
    p = loo_predict(X, y, n_components)
    return np.sqrt(((p - y.ravel()) ** 2).mean())


N_PERM = 500
yrand = {}
for n in (1, N_COMPONENTS):
    observed = loo_rmse(X, y, n)
    rng = np.random.default_rng(RANDOM_SEED)
    perm = np.array([loo_rmse(X, rng.permutation(y.ravel()).reshape(-1, 1), n)
                     for _ in range(N_PERM)])
    p_value = (perm <= observed).mean()
    yrand[n] = {"observed": observed, "perm": perm, "p": p_value}
    print(f"{n} component(s): observed RMSE_LOO = {observed:.1f}%, "
          f"permuted = {perm.mean():.1f} +/- {perm.std(ddof=1):.1f}%, p = {p_value:.3f}")

## 4. Figure S1

* **(a)** parity plot of the cross-validated predictions for the one-component model
* **(b)** RMSE and Q2 as a function of the number of latent variables
* **(c)** distribution of the permuted errors compared with the unpermuted model

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

# (a) parity plot, one-component model
ax = axes[0]
ax.scatter(y_obs, loo_preds[1], color="#232A34", zorder=3)
for name, xo, yp in zip(train["name"], y_obs, loo_preds[1]):
    ax.annotate(name, (xo, yp), fontsize=7, xytext=(3, 3), textcoords="offset points")
lim = (-10, 115)
ax.plot(lim, lim, ls="--", color="grey", lw=1)
ax.set(xlim=lim, ylim=lim, xlabel="Observed yield [%]",
       ylabel="LOO predicted yield [%]", title="(a) 1-component model")

# (b) error and Q2 versus number of latent variables
ax = axes[1]
n_list = list(range(1, 6))
rmse_list = [np.sqrt(((loo_preds[n] - y_obs) ** 2).mean()) for n in n_list]
q2_list = [1 - ((loo_preds[n] - y_obs) ** 2).sum() / tss for n in n_list]
best = int(np.argmin(rmse_list))
ax.plot(n_list, rmse_list, "o-", color="#232A34", label="RMSE$_{LOO}$")
ax.plot(n_list[best], rmse_list[best], "o", color="#FF2E62", ms=10, zorder=4)
ax.axhline(null_rmse, ls=":", color="grey")
ax.text(5, null_rmse + 0.4, "null model", ha="right", fontsize=8, color="grey")
ax.set(xlabel="Number of latent variables", ylabel="RMSE$_{LOO}$ [%]",
       xticks=n_list, title="(b) Cross-validated error")
ax2 = ax.twinx()
ax2.plot(n_list, q2_list, "s--", color="#FF2E62")
ax2.set_ylabel("$Q^2$", color="#FF2E62")
ax2.tick_params(axis="y", colors="#FF2E62")

# (c) y-randomization
ax = axes[2]
ax.hist(yrand[1]["perm"], bins=30, color="grey", alpha=0.8)
ax.axvline(yrand[1]["observed"], color="#FF2E62", lw=2)
ax.set(xlabel="RMSE$_{LOO}$ [%]", ylabel="Count",
       title=f"(c) y-randomization (p = {yrand[1]['p']:.3f})")

fig.tight_layout()
fig.savefig(OUT / "figure_s1.png", dpi=600, bbox_inches="tight")
plt.show()

## Summary

The model is not a chance correlation: the cross-validated error is well below the
null model and only a small fraction of the y-randomized models reach it. The
cross-validated error of about 30% is nevertheless too large for quantitative
prediction of individual yields, which is why the model is used for **rank-ordering**
candidate solvents rather than for absolute prediction.